# StormScope-EU `true_color` with a geostationary grid mapping

Test for [TILES_BUG.md](TILES_BUG.md). Flux tiles for `gtc-berlin/stormscope-eu` / `stormscope_v2`
stop at ~50°N because the renderer drops every grid row that has a NaN in the 2-D `lat`/`lon`
coordinates. The top-left and top-right corners of the FCI crop are off the Earth's disk, so
those rows all have NaNs.

Suggested fix (Earthmover): because this is geostationary data, store a CF **grid mapping
variable** with the geostationary projection so the renderer works from the 1-D `y`/`x`
scan angles and never touches 2-D lat/lon. This is the same layout as
[`earthmover-public/goes-16` / `ABI-L2-MCMIPF/post-2023-04-19`](https://app.earthmover.io/earthmover-public/goes-16/tree/main/ABI-L2-MCMIPF/post-2023-04-19)
(`goes_imager_projection`). Earthmover also suggested the `polygons` tile style might cope better than `raster`; measured, it doesn't (see `docs/WEB_TILES_SERVICE_REQUIREMENTS.md`).

This notebook reads `stormscope_v2` from `main` and writes **only `true_color`, first 2 lead times**,
to the group `stormscope_v2` on a new branch **`tiles_drop_rows`**. `main` is untouched.

In [ ]:
import arraylake as al
import numpy as np
import pyproj
import xarray as xr
from icechunk.xarray import to_icechunk

REPO = "gtc-berlin/stormscope-eu"
GROUP = "stormscope_v2"
SOURCE_BRANCH = "main"
BRANCH = "tiles_drop_rows"
N_LEADS = 2  # lead_time 0 (observed analysis) and +10 min

repo = al.Client().get_repo(REPO)
src = xr.open_zarr(
    repo.readonly_session(SOURCE_BRANCH).store, group=GROUP, consolidated=False
)
src

## Projection parameters

MTG-I1 FCI, full disk at 0°E. Constants are the ones earth2studio's `MeteosatFCI` uses to
compute the stored `lat`/`lon` (`earth2studio/data/meteosat_fci.py`).

Two FCI details differ from GOES:

- **`sweep_angle_axis = "y"`**. GOES uses `"x"`. With `"x"` the recomputed coordinates are
  off by up to 0.35°.
- **`x` is positive to the west** in the FCI/earth2studio convention (`x[0]` = +0.052 rad is
  the western edge, at about 26°W). CF/PROJ `geos` expects `projection_x_coordinate`
  positive to the east, so `x` is negated below. Without that, longitudes come out mirrored.

In [ ]:
H = 35786400.0  # perspective_point_height, m above the ellipsoid
GRID_MAPPING = "fci_geos_projection"

gm_attrs = {
    "long_name": "MTG-I1 FCI fixed grid projection",
    "grid_mapping_name": "geostationary",
    "perspective_point_height": H,
    "semi_major_axis": 6378137.0,
    "semi_minor_axis": 6356752.3142,
    "inverse_flattening": 298.257223563,
    "latitude_of_projection_origin": 0.0,
    "longitude_of_projection_origin": 0.0,
    "sweep_angle_axis": "y",
}
crs = pyproj.CRS.from_cf(gm_attrs)
gm_attrs["crs_wkt"] = crs.to_wkt()  # for rioxarray / GDAL readers
crs

### Check against the stored lat/lon

Transform the scan angles (negated `x`, times `H`) to lon/lat and compare them with the 2-D
`lat`/`lon` in the repo. The difference should be ~1e-6° and the off-disk NaN mask should match exactly.

In [ ]:
x_east = -src.x.values  # radians, positive east
y = src.y.values  # radians, positive north

X, Y = np.meshgrid(x_east * H, y * H)
lon_p, lat_p = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(X, Y)
lon_p[~np.isfinite(lon_p)] = np.nan
lat_p[~np.isfinite(lat_p)] = np.nan

lat, lon = src.lat.values, src.lon.values
print("max |dlat| :", np.nanmax(np.abs(lat_p - lat)))
print("max |dlon| :", np.nanmax(np.abs(lon_p - lon)))
print("NaN mask identical:", np.array_equal(np.isnan(lat_p), np.isnan(lat)))
assert np.nanmax(np.abs(lat_p - lat)) < 1e-4 and np.nanmax(np.abs(lon_p - lon)) < 1e-4

## Build the subset

- `true_color`, `lead_time` 0 and 10 min.
- `x` is replaced by its east-positive version, and `x`/`y` get CF projection-coordinate attrs
  (radians, as in GOES).
- The 2-D `lat`/`lon` are **dropped** so that the grid mapping is the only georeference and the
  renderer can't fall back to the NaN-laden 2-D coordinates. That fallback is exactly the bug
  being tested. The GOES-16 dataset has no 2-D lat/lon either.
- `rgb` stays a string coordinate. `raster/rgb` reads the three bands from it.
- The source encoding is cleared (it still carries `coordinates: "lat lon"`), and each field is one chunk.

In [ ]:
ds = (
    src[["true_color"]]
    .isel(lead_time=slice(0, N_LEADS))
    .drop_vars(["lat", "lon"])
    .assign_coords(x=("x", x_east))
)

ds["x"].attrs = {
    "units": "rad",
    "axis": "X",
    "standard_name": "projection_x_coordinate",
    "long_name": "FCI fixed grid projection x-coordinate (scan angle, positive east)",
}
ds["y"].attrs = {
    "units": "rad",
    "axis": "Y",
    "standard_name": "projection_y_coordinate",
    "long_name": "FCI fixed grid projection y-coordinate (scan angle, positive north)",
}
ds[GRID_MAPPING] = xr.DataArray(np.int32(0), attrs=gm_attrs)
ds["true_color"].attrs = {
    **src.true_color.attrs,
    "grid_mapping": GRID_MAPPING,
    "units": "1",
}
ds.attrs = {
    **src.attrs,
    "Conventions": "CF-1.8",
    "comment": (
        f"Subset of {REPO}/{GROUP}@{SOURCE_BRANCH}: true_color, first {N_LEADS} lead times, "
        "georeferenced by a geostationary grid mapping instead of 2-D lat/lon (see TILES_BUG.md)."
    ),
}

for v in ds.variables.values():
    v.encoding = {}
encoding = {
    "true_color": {
        "chunks": (1, 1, 1, ds.sizes["y"], ds.sizes["x"], ds.sizes["rgb"])
    }
}
ds

## Write to branch `tiles_drop_rows`

The branch is created from the tip of `main` if it doesn't exist. `mode="w"` replaces the
`stormscope_v2` group on that branch, so a re-run overwrites it cleanly. Other groups on the
branch are inherited from `main` and stay as they are.

In [ ]:
if BRANCH not in repo.list_branches():
    repo.create_branch(BRANCH, repo.lookup_branch(SOURCE_BRANCH))

session = repo.writable_session(BRANCH)
to_icechunk(ds.load(), session, group=GROUP, mode="w", encoding=encoding)
snapshot = session.commit(
    f"{GROUP}: true_color, {N_LEADS} lead times, geostationary grid mapping, no 2-D lat/lon"
)
print(BRANCH, "->", snapshot)
print(f"https://app.earthmover.io/{REPO}/tree/{BRANCH}/{GROUP}")

## Read it back

Reopen from the branch and check that the grid mapping round-trips, that `true_color` refers to it,
and that Berlin (52.52°N, 13.40°E, well north of the old ~50°N cutoff) has data in both lead times.

In [ ]:
out = xr.open_zarr(
    repo.readonly_session(BRANCH).store, group=GROUP, consolidated=False
)
print(out)
print()
print("grid_mapping:", out.true_color.attrs["grid_mapping"])
crs_rt = pyproj.CRS.from_cf(out[GRID_MAPPING].attrs)
print(crs_rt.to_proj4())

bx, by = pyproj.Transformer.from_crs("EPSG:4326", crs_rt, always_xy=True).transform(13.40, 52.52)
berlin = out.true_color.sel(x=bx / H, y=by / H, method="nearest").isel(ensemble=0, time=0)
print(berlin.compute().values)
assert np.isfinite(berlin.values).all()

In [ ]:
out.true_color.isel(ensemble=0, time=0, lead_time=0).plot.imshow(
    rgb="rgb", robust=True, size=5, aspect=ds.sizes["x"] / ds.sizes["y"]
);

## Tiles

Point the Flux tiles service at the branch (dataset path `stormscope-eu/tiles_drop_rows/stormscope_v2`)
and compare a tile north of 50°N (Berlin, `WebMercatorQuad/7/41/68`) with the same tile from `main`:

```
.../tiles/WebMercatorQuad/7/41/68?style=raster/rgb&variables=true_color&colorscalerange=0,1&ensemble=0&lead_time=0 minutes
```

`style=polygons/rgb` gives the same result as `raster/rgb`: with the grid mapping both render, and with 2-D lat/lon only both still cut off at ~50°N.